# RAG Notebook (Code Only)

Extracted from HTML export.

In [5]:
import re
from openai import OpenAI

raw = open("openaikey.txt", "r").read()
m = re.search(r"(sk-[A-Za-z0-9\-_]+)", raw)
if not m:
    raise RuntimeError("No API key found in openaikey.txt — put the raw key or adjust parsing.")
api_key = m.group(1)
#print("Using key:", repr(api_key))   # sanity check
client = OpenAI(api_key=api_key)

resp = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[{"role":"user","content":"ping"}]
)
print(resp.choices[0].message.content)


Pong! How can I assist you today?


In [8]:
import PyPDF2
import faiss
import numpy as np
from openai import OpenAI


In [9]:
def read_pdf(path):
    reader = PyPDF2.PdfReader(path)
    text = ""
    for page in reader.pages:
        page_text = page.extract_text() # 1page - text , 2 page - image , 3 - text
        if page_text:
            text = text+page_text
    return text

text = read_pdf("Meropenem_Information.pdf")
print(text)
print(len(text))


Meropenem
Information About Meropenem
Meropenem is a broad-spectrum antibiotic used to treat severe bacterial infections and to prevent infection after
surgery of the colon or rectum.
Uses of Meropenem
It is used to treat infections of the skin and soft tissues, urinary tract, blood, brain (meningitis), lungs (pneumonia),
abdomen, and infections during or after delivery.
How Meropenem Works
Meropenem kills bacteria by preventing the formation of the bacterial cell wall, which is essential for their survival.
Common Side Effects
Rash, headache, nausea, vomiting, diarrhea, itching, constipation, injection site inflammation, anemia, sepsis,
apnea, and shock.
Expert Advice
Meropenem is usually administered in a hospital setting via intravenous infusion over 20 to 60 minutes. Inform
your doctor if you are allergic to penicillin or taking seizure medication. Kidney, liver, and blood counts may be
monitored during treatment.
Important Safety Information
Meropenem is effective only against bac

In [27]:
def make_chunks(text , chunk_size = 500):
    chunkslist = []
    start = 0

    while start<len(text): #
        end = start+chunk_size # 
        chunk = text[start:end] 
        chunkslist.append(chunk)
        start = end
    return chunkslist


In [28]:
def get_embeddings(chunks):
    embeddings = []

    for chunk in chunks:
        response = client.embeddings.create(input = chunk ,model="text-embedding-3-small" )
        vector = response.data[0].embedding
        embeddings.append(vector)
    return np.array(embeddings).astype("float32")


In [29]:
def build_faiss_index(embeddings):
    dimension = len(embeddings[0]) # 1536
    index = faiss.IndexFlatL2(dimension)
    index.add(embeddings)
    return index


index = build_faiss_index(em)


In [30]:
def search_chunks(question, chunks , index):
    # Converting the Question into embeddings
    q_embed = client.embeddings.create(input=question,model="text-embedding-3-small").data[0].embedding
    q_embed = np.array([q_embed]).astype("float32")

    #searching the vector index and find the top two index which matches embeddings of our question
    distaces , indices = index.search(q_embed,k=2)  # [[12,14],"hello"]

    #find the respective chunks of the index returned
    best_chunks = [chunks[i] for i in indices[0]]
    return best_chunks


In [31]:
name = "Ram"
age = 10
print(f"My name is {name} and my age is {age}")


My name is Ram and my age is 10


In [33]:
def ask_question(c,q):
    context = c
    question = q
    prompt = f"Answer the question based on this text \n{context} \n\n Question:{question}"
    print(prompt)


In [34]:
context = "Python is created in 1989 .It came before JAVA."
question = "when python is created"
ask_question(context,question)


Answer the question based on this text 
Python is created in 1989 .It came before JAVA. 

 Question:when python is created


In [37]:
def ask_question(chunks , question):
    context = "\n\n".join(chunks)
    prompt = f"Answer the question based on this text \n{context} \n\n {question}"
    #llm

    response = client.chat.completions.create(model = "gpt-4o-mini",messages = [{"role":"user","content":prompt}])
    return response.choices[0].message.content


#chuk1

#chunk2 


In [ ]:
# ------------------------------------------------------------
# 📘 SIMPLE PROJECT: Ask Questions from a PDF using OpenAI + FAISS
# ------------------------------------------------------------
# This project will:
# 1. Read text from a PDF file
# 2. Split it into small parts (chunks)
# 3. Convert text into numbers (embeddings)
# 4. Store them in a FAISS database
# 5. Take a question from the user
# 6. Find the most similar text parts(chunk) from the PDF
# 7. Send them to OpenAI to get a final answer
# ------------------------------------------------------------

# 🧩 Step 1: Import libraries
import PyPDF2                # to read the PDF
import faiss                 # to store and search embeddings
import numpy as np           # for handling numeric data
from openai import OpenAI    # for using OpenAI models

# Create OpenAI client (make sure your API key is set)
#client = OpenAI()

# 🧠 Step 2: Function to read all text from a PDF
def read_pdf(path):
    reader = PyPDF2.PdfReader(path)   # open the PDF file
    text = ""                         # empty string to store all text
    for page in reader.pages:         # go through each page
        page_text = page.extract_text()  # extract text from page
        if page_text:                    # make sure it's not empty
            text = text + page_text      # add it to full text
    return text  # 7000 characters


# 🪓 Step 3: Function to break the text into smaller parts
# Why? Because models can’t handle very long text all at once.
def make_chunks(text, chunk_size=500):
    chunks = []                    # list to store text pieces
    start = 0                      # start index 
    while start < len(text):       # loop until end of text
        end = start + chunk_size   # calculate end index  
        chunk = text[start:end]    # get a piece of text  text[500:1000]
        chunks.append(chunk)       # add to list
        start = end                # move to next chunk
    return chunks


# 🔢 Step 4: Function to convert text chunks into embeddings (numbers)
def get_embeddings(chunks):
    embeddings = []    # will store all embeddings
    for chunk in chunks:
        response = client.embeddings.create(
            input=chunk,
            model="text-embedding-3-small"  # small & fast model
        )
        vector = response.data[0].embedding  # get the embedding vector
        embeddings.append(vector)
    return np.array(embeddings).astype("float32")  # convert to numpy array


# 💾 Step 5: Function to create and store vectors in FAISS
def build_faiss_index(embeddings):
    dimension = len(embeddings[0])     # number of numbers in each vector
    index = faiss.IndexFlatL2(dimension)  # L2 = normal distance measure
    index.add(embeddings)              # store all vectors in FAISS
    return index


# ❓ Step 6: Function to search for the most similar text
def search_chunks(question, chunks, index):
    # Convert the question into embedding (numbers)
    q_embed = client.embeddings.create(
        input=question,
        model="text-embedding-3-small"
    ).data[0].embedding

    q_embed = np.array([q_embed]).astype("float32")  # convert to numpy array

    # Search in FAISS for the most similar text chunk
    distances, indices = index.search(q_embed, k=2)  # get top 2 closest chunks
    best_chunks = [chunks[i] for i in indices[0]]    # pick those chunks
    return best_chunks


# 💬 Step 7: Function to ask OpenAI for the final answer
def ask_question(chunks, question):
    context = "\n\n".join(chunks)  # join chunks as context
    prompt = f"Answer the question based on this text:\n{context}\n\nQuestion: {question}"

    response = client.chat.completions.create(
        model="gpt-4o-mini",  # small but smart model
        messages=[{"role": "user", "content": prompt}]
    )

    return response.choices[0].message.content


# 🚀 Step 8: Main function to run everything
def main():
    # 1. Read text from a sample PDF
    text = read_pdf("medicine.pdf")
    print("✅ PDF Loaded.")

    # 2. Split text into smaller chunks
    chunks = make_chunks(text)
    print("✅ Text divided into chunks.")

    # 3. Create embeddings for all chunks
    embeddings = get_embeddings(chunks)
    print("✅ Embeddings created.")

    # 4. Store embeddings in FAISS
    index = build_faiss_index(embeddings)
    print("✅ FAISS index created.")

    # 5. Ask user for a question
    question = input("Ask your Question from PDF : ")

    # 6. Find similar text chunks
    best_chunks = search_chunks(question, chunks, index)
    print("✅ Found related text parts.")
    print(best_chunks)
    print("******************************")

    # 7. Get final answer from OpenAI
    answer = ask_question(best_chunks, question)
    print("\n🧠 Answer from OpenAI:")
    print(answer)


# Run the program
if __name__ == "__main__":
    main()

#token cost # 5 dollars - questions,answer , documents
#vector -db

    
# Meropenem uses
#list down the uses of Meropenem 
#what are side effects of Meropenem 
#what are processors in computers
#is vomiting a side effect of Meropenem

#gibili - internal training 
#own app to crete gibili - photo(not used for internal training)
